# Aula 8 — Integração do pipeline

Agora vamos retomar todas as pastas e verificar a cadeia completa de arquivos.

Esta aula não precisa instalar uma nova ferramenta externa: o foco é rastrear entradas,
saídas e métricas usando Python.

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

In [ ]:
print("\nPastas existentes na raiz:")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print(" -", p.name)

## 2. Verificar os produtos das aulas anteriores

In [ ]:
paths = {
    "R1_raw": PASTAS["03_raw"] / f"{RUN}_1.fastq.gz",
    "R2_raw": PASTAS["03_raw"] / f"{RUN}_2.fastq.gz",
    "R1_trim": PASTAS["04_trimmed"] / f"{SAMPLE}_R1_paired.fastq.gz",
    "R2_trim": PASTAS["04_trimmed"] / f"{SAMPLE}_R2_paired.fastq.gz",
    "contigs": PASTAS["05_contigs"] / f"{SAMPLE}.contigs.fasta",
    "uce_db": PASTAS["06_results"] / "probe.matches.sqlite",
    "uce_fasta": PASTAS["07_taxon_sets"] / "all-taxa-incomplete.fasta",
}

for nome, path in paths.items():
    print(f"{nome:10s}", "OK" if path.exists() else "AUSENTE", path)

## 3. Funções de contagem

In [ ]:
import gzip

def nreads(path):
    if not path.exists():
        return None
    with gzip.open(path, "rt") as f:
        return sum(1 for _ in f) // 4

def fasta_lengths(path):
    if not path.exists():
        return []
    lengths, seq = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    lengths.append(len("".join(seq)))
                seq = []
            else:
                seq.append(line)
        if seq:
            lengths.append(len("".join(seq)))
    return lengths

def n50(lengths):
    if not lengths:
        return None
    half = sum(lengths) / 2
    acc = 0
    for L in sorted(lengths, reverse=True):
        acc += L
        if acc >= half:
            return L

## 4. Calcular métricas do projeto

In [ ]:
raw = nreads(paths["R1_raw"])
trim = nreads(paths["R1_trim"])
contig_lengths = fasta_lengths(paths["contigs"])
uce_lengths = fasta_lengths(paths["uce_fasta"])

metrics = {
    "run": RUN,
    "sample": SAMPLE,
    "reads_R1_raw": raw,
    "reads_R1_paired_trimmed": trim,
    "retencao_pct": round(trim/raw*100, 2) if raw and trim is not None else None,
    "n_contigs_min200": len(contig_lengths),
    "assembly_total_bp": sum(contig_lengths) if contig_lengths else None,
    "assembly_max_contig": max(contig_lengths) if contig_lengths else None,
    "assembly_N50": n50(contig_lengths),
    "n_uce_records": len(uce_lengths),
    "uce_mean_length": round(sum(uce_lengths)/len(uce_lengths), 1) if uce_lengths else None,
}

metrics

## 5. Criar e salvar o resumo

In [ ]:
import pandas as pd

summary = pd.DataFrame([metrics])
display(summary.T)

CSV = PASTAS["08_integracao"] / f"resumo_pipeline_{RUN}.csv"
summary.to_csv(CSV, index=False)

print("Resumo salvo em:", CSV)

## 6. Reconstruir o fluxo

Para cada seta, indique o arquivo de entrada e o arquivo de saída:

**SRA → FASTQ → FastQC/MultiQC → Trimmomatic → SPAdes → contigs → PHYLUCE → loci UCE**

Perguntas finais:
1. Em qual etapa deixamos de trabalhar com reads e passamos a contigs?
2. Em qual etapa as probes entram no pipeline?
3. Que arquivos persistiram no Drive entre as aulas?
4. Que componentes do runtime precisaram ser reinstalados?
5. Por que dados, ambiente, parâmetros e caminhos fazem parte da reprodutibilidade?